In [ ]:

import sys
sys.path.insert(0, "./NeuroLM")

from model.model_vq import VQ
from model.model_neural_transformer import NTConfig
from Util.ModelSurgery.neurolmvq_to_customfft import patch_vq
from Util.DataLoader.ExtractedMAT_FFT.loader import create_distributed_dataloaders, FFTDataLoader

import torch

ModuleNotFoundError: No module named 'transformers'

In [4]:
!pip3 


Usage:   
  pip3 <command> [options]

Commands:
  install                     Install packages.
  download                    Download packages.
  uninstall                   Uninstall packages.
  freeze                      Output installed packages in requirements format.
  inspect                     Inspect the python environment.
  list                        List installed packages.
  show                        Show information about installed packages.
  check                       Verify installed packages have compatible dependencies.
  config                      Manage local and global configuration.
  search                      Search PyPI for packages.
  cache                       Inspect and manage pip's wheel cache.
  index                       Inspect information available from package indexes.
  wheel                       Build wheels from your requirements.
  hash                        Compute hashes of package archives.
  completion                  A helper c

In [ ]:
import sys
print(sys.executable)

!which python
!which pip3
!{sys.executable} -m pip show transformers

In [ ]:
vq_path = ".weights/NeuroLm/checkpoints/VQ.pt"
vq = torch.load(vq_path, map_location="cpu", weights_only=False)

In [ ]:
encoder_config = NTConfig(**vq["encoder_args"])
decoder_config = NTConfig(**vq["decoder_args"])

model = VQ(
    encoder_config=encoder_config,
    decoder_config=decoder_config
)

In [ ]:
loader = FFTDataLoader("Data/MAT_CHUNK_EXTRACTED")

In [ ]:
state_dict = vq["model"]

clean_state_dict = {}

for k, v in state_dict.items():
    if k.startswith("_orig_mod.VQ."):
        new_k = k[len("_orig_mod.VQ."):]
        clean_state_dict[new_k] = v
        
model.load_state_dict(clean_state_dict)
model = patch_vq(model, fft_dim=101, n_channels=128)